# Continuous Safe RL Baselines

This tutorial runs a small, reproducible continuous-action ablation in MASA. We use `ContinuousCartpole` and compare unconstrained PPO, constrained policy optimization (CPO), and PPO with a learned Lagrangian penalty. Increase `TRAINING_STEPS` and `SEEDS` for a research experiment.


## CPU-first setup

Keep the notebook portable before importing MASA/JAX modules.


In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")


## Experiment design

`ContinuousCartpole` has a one-dimensional `Box` action space. The CMDP cost is `1.0` when the cart-pole leaves the stable set and `0.0` otherwise. All algorithms use the same horizon, seed, rollout length, and network architecture. This is a smoke ablation, not a statistically significant comparison.


In [ ]:
from pathlib import Path
import sys

repo_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "masa").is_dir()
)
sys.path.insert(0, str(repo_root))

from masa.common.utils import make_env
from masa.envs.continuous.cartpole import cost_fn, label_fn
from masa.algorithms.on_policy import CPO, PPO, PPOLag

ENV_ID = "ContinuousCartpole"
MAX_EPISODE_STEPS = 25
TRAINING_STEPS = 512
ROLLOUT_STEPS = 128
SEEDS = [0]
LOG_ROOT = Path("runs/tutorial_07")


def make_continuous_safe_env(max_episode_steps=MAX_EPISODE_STEPS):
    return make_env(
        ENV_ID,
        "CMDP",
        max_episode_steps,
        label_fn=label_fn,
        constraint_kwargs={"cost_fn": cost_fn, "budget": 0.0},
    )


COMMON_CONFIG = {
    "learning_rate": 3e-4,
    "n_steps": ROLLOUT_STEPS,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "ent_coef": 0.0,
    "vf_coef": 1.0,
    "max_grad_norm": 0.5,
    "policy_kwargs": {"net_arch": [32, 32], "log_std_init": 0.0},
}

ALGORITHM_CONFIGS = {
    "PPO": {**COMMON_CONFIG, "batch_size": 64, "n_epochs": 2},
    "CPO": {
        **COMMON_CONFIG,
        "learning_rate": 1e-3,
        "n_critic_updates": 2,
        "target_kl": 0.01,
        "cg_iters": 10,
        "cg_damping": 0.1,
        "fvp_sample_freq": 1,
        "line_search_steps": 10,
        "line_search_decay": 0.8,
        "normalize_reward_advantages": True,
        "normalize_cost_advantages": True,
        "cost_limit": 0.0,
        "cost_gamma": 0.99,
        "cost_gae_lambda": 0.95,
    },
    "PPO-Lagrangian": {
        **COMMON_CONFIG,
        "batch_size": 64,
        "n_epochs": 2,
        "normalize_reward_advantages": True,
        "normalize_cost_advantages": True,
        "cost_limit": 0.0,
        "cost_gamma": 0.99,
        "cost_gae_lambda": 0.95,
        "lagrangian_multiplier_init": 0.0,
        "lambda_lr": 0.01,
        "lagrangian_upper_bound": 100.0,
    },
}

ALGORITHMS = {"PPO": PPO, "CPO": CPO, "PPO-Lagrangian": PPOLag}

probe_env = make_continuous_safe_env()
obs, info = probe_env.reset(seed=0)
print("observation_space:", probe_env.observation_space)
print("action_space:", probe_env.action_space)
print("initial labels:", sorted(info["labels"]))
print("initial constraint:", info["constraint"])
assert probe_env.action_space.shape == (1,)
probe_env.close()


## Train and evaluate each baseline

The training API is shared by all three algorithms. Evaluation uses deterministic actions and reports return, cumulative cost, violation rate, and episode length.


In [ ]:
import jax.random as jr
import numpy as np


def evaluate_policy(model, episodes=3, seed=10_000):
    env = make_continuous_safe_env()
    returns, costs, violations, lengths = [], [], [], []
    for episode in range(episodes):
        obs, _ = env.reset(seed=seed + episode)
        done = False
        episode_return = 0.0
        episode_cost = 0.0
        episode_violations = 0
        length = 0
        while not done:
            action = model.act(
                jr.PRNGKey(seed + episode + length), obs, deterministic=True
            )
            obs, reward, terminated, truncated, info = env.step(action)
            episode_return += float(reward)
            step_cost = float(info["constraint"]["step"]["cost"])
            episode_cost += step_cost
            episode_violations += int(step_cost > 0.0)
            length += 1
            done = terminated or truncated
        returns.append(episode_return)
        costs.append(episode_cost)
        violations.append(episode_violations / max(length, 1))
        lengths.append(length)
    env.close()
    return {
        "return_mean": float(np.mean(returns)),
        "cost_mean": float(np.mean(costs)),
        "violation_rate": float(np.mean(violations)),
        "length_mean": float(np.mean(lengths)),
    }


def run_ablation(seed):
    results = []
    for name, algorithm in ALGORITHMS.items():
        train_env = make_continuous_safe_env()
        model = algorithm(
            train_env,
            tensorboard_logdir=str(
                LOG_ROOT / f"{name.lower().replace('-', '_')}_s{seed}"
            ),
            seed=seed,
            device="cpu",
            verbose=0,
            env_fn=make_continuous_safe_env,
            **ALGORITHM_CONFIGS[name],
        )
        model.train(
            TRAINING_STEPS,
            num_eval_episodes=0,
            eval_freq=0,
            log_freq=0,
            stats_window_size=10,
        )
        metrics = evaluate_policy(model, seed=10_000 + seed * 100)
        results.append({"algorithm": name, "seed": seed, **metrics})
        train_env.close()
    return results


RESULTS = [row for seed in SEEDS for row in run_ablation(seed)]
RESULTS


## Compare results

With the default `cost_budget=0.0`, lower cost and violation rate are safer. The short run verifies the end-to-end continuous-action path; use multiple seeds and substantially more training steps before drawing algorithmic conclusions.


In [ ]:
from IPython.display import Markdown, display


def markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = [
        "| " + " | ".join(str(row[column]) for column in columns) + " |"
        for row in rows
    ]
    return "\n".join([header, divider, *body])


TABLE_COLUMNS = [
    "algorithm",
    "seed",
    "return_mean",
    "cost_mean",
    "violation_rate",
    "length_mean",
]
display(Markdown(markdown_table(RESULTS, TABLE_COLUMNS)))


## Extending the ablation

For a research-quality comparison, set `SEEDS = [0, 1, 2, 3, 4]`, increase `TRAINING_STEPS`, and aggregate confidence intervals over seeds. Keep the environment, budget, evaluation seeds, and network architecture fixed across PPO, CPO, and PPO Lagrangian (implemented as `PPO-Lagrangian`).
